Week 15 · Day 2 — Vector Embeddings & In-Memory Index
Why this matters

Retrieval in RAG depends on vector embeddings: numerical representations of text that capture meaning. Once you have them, you can do semantic search: “find texts most similar in meaning,” not just exact keyword matches.

Theory Essentials

Embedding model: maps text → high-dim vector (e.g., 384–768 dims).

SentenceTransformers: library with many pretrained embedding models.

Similarity metric: usually cosine similarity between vectors.

Index: store embeddings so queries can be compared fast.

In-memory index: simplest version — just store vectors in a list/array.

In [1]:
# Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Knowledge base
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London."
]

# 3. Embed documents
embeddings = model.encode(docs)

# 4. Query
query = "Where is the Colosseum located?"
q_emb = model.encode([query])

# 5. Compute similarity
sims = cosine_similarity(q_emb, embeddings)[0]

# 6. Retrieve top-3 docs
top3 = np.argsort(sims)[-3:][::-1]
print("Query:", query)
print("\nTop Retrieved Docs:")
for i in top3:
    print(f"{docs[i]}  (score={sims[i]:.3f})")


Query: Where is the Colosseum located?

Top Retrieved Docs:
The Colosseum is in Rome.  (score=0.842)
The Prado Museum is in Madrid.  (score=0.274)
The Brandenburg Gate is in Berlin.  (score=0.220)


1) Core (10–15 min)

Task: Change the query to “Where is Big Ben located?” and retrieve the most similar doc.

In [2]:
query = "Where is Big Ben located?"
q_emb = model.encode([query])
sims = cosine_similarity(q_emb, embeddings)[0]
print(docs[np.argmax(sims)])


Big Ben is in London.


2) Practice (10–15 min)

Task: Write a function search(query, docs, k=2) that returns the top-k documents.

In [3]:
def search(query, docs, k=2):
    q_emb = model.encode([query])
    sims = cosine_similarity(q_emb, embeddings)[0]
    topk = np.argsort(sims)[-k:][::-1]
    return [(docs[i], sims[i]) for i in topk]

print(search("What city has the Prado Museum?", docs, k=2))


[('The Prado Museum is in Madrid.', np.float32(0.849714)), ('The Colosseum is in Rome.', np.float32(0.27446193))]


3) Stretch (optional, 10–15 min)

Task: Add a new document (“The Acropolis is in Athens.”). Re-embed and test with query

In [4]:
docs.append("The Acropolis is in Athens.")
embeddings = model.encode(docs)
print(search("Where is the Acropolis?", docs, k=1))


[('The Acropolis is in Athens.', np.float32(0.89424664))]


Mini-Challenge (≤40 min)

Build a Mini Search Engine

Write a function build_index(docs) that:

Stores documents and embeddings.

Returns a search function that retrieves top-3 docs for any query.

Acceptance Criteria:

Encapsulated index (docs + embeddings).

Can answer at least 3 queries correctly.

Retrieval works even after adding new documents

In [5]:
def build_index(docs):
    """Store docs + embeddings and return tiny search + add helpers."""
    embeddings = model.encode(docs)

    def search(query, k=3):
        q_emb = model.encode([query])
        sims = cosine_similarity(q_emb, embeddings)[0]
        topk = np.argsort(sims)[-k:][::-1]
        return [(docs[i], float(sims[i])) for i in topk]

    def add(new_docs):
        # allow adding docs later; re-embed (simple but fine for mini-challenge)
        nonlocal docs, embeddings
        docs += list(new_docs)
        embeddings = model.encode(docs)

    return search, add

# 1) Build index
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London.",
    "Sagrada Familia is in Barcelona."
]
search, add = build_index(docs)

# 2) At least 3 queries
queries = [
    "Where is the Colosseum located?",
    "In which city is Big Ben?",
    "What city has the Prado Museum?"
]

for q in queries:
    print("\nQ:", q)
    for text, score in search(q, k=3):
        print(f" - {text}  (score={score:.3f})")

# 3) Add new docs, retrieval still works
add(["The Acropolis is in Athens."])
print("\nQ: Where is the Acropolis?")
for text, score in search("Where is the Acropolis?", k=3):
    print(f" - {text}  (score={score:.3f})")



Q: Where is the Colosseum located?
 - The Colosseum is in Rome.  (score=0.842)
 - The Prado Museum is in Madrid.  (score=0.274)
 - Sagrada Familia is in Barcelona.  (score=0.269)

Q: In which city is Big Ben?
 - Big Ben is in London.  (score=0.857)
 - Sagrada Familia is in Barcelona.  (score=0.209)
 - The Eiffel Tower is in Paris.  (score=0.159)

Q: What city has the Prado Museum?
 - The Prado Museum is in Madrid.  (score=0.850)
 - The Colosseum is in Rome.  (score=0.274)
 - Sagrada Familia is in Barcelona.  (score=0.233)

Q: Where is the Acropolis?
 - The Acropolis is in Athens.  (score=0.894)
 - The Colosseum is in Rome.  (score=0.454)
 - Sagrada Familia is in Barcelona.  (score=0.313)


Notes / Key Takeaways

Embeddings = semantic fingerprints of text.

Cosine similarity finds “closest” meanings.

Index = store vectors for retrieval.

Today: simple Python list index; later we’ll use FAISS/Chroma for scale.

RAG effectiveness hinges on embedding quality + indexing method.

Reflection

Why do we prefer cosine similarity over raw dot product?

How would retrieval change if docs were very long (paragraphs vs sentences)?

Why do we prefer cosine similarity over raw dot product?

Dot product is affected by the magnitude (length) of vectors, so longer sentences or embeddings with larger norms might look “more similar” just because they’re bigger.

Cosine similarity normalizes for length and only measures the angle between vectors → a better measure of semantic closeness.



How would retrieval change if docs were very long (paragraphs vs sentences)?

Long docs may contain many topics, so a single embedding might blur meanings and reduce retrieval accuracy.

Queries could match irrelevant parts inside a long paragraph.

In practice, we chunk documents (e.g., split paragraphs into sentences or small windows) before embedding to keep retrieval focused.